In [1]:
# Cell 1: Khai báo thư viện
print('Đang khai báo thư viện')

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.utils.class_weight import compute_sample_weight
import matplotlib.pyplot as plt
import seaborn as sns

print('Khai báo thư viện thành công')

Đang khai báo thư viện
Khai báo thư viện thành công


In [2]:
# Cell 2: Khai báo dữ liệu
print('Khai báo dữ liệu')

train_path = os.path.join("..", "Data", "DataCleaned", "data_train_cleaned.csv")
val_path = os.path.join("..", "Data", "DataCleaned", "validation_cleaned.csv")

df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)

print("THÔNG TIN DỮ LIỆU")
print(f"Train shape: {df_train.shape}")
print(f"Validation shape: {df_val.shape}")
print(f"\nTrain columns:\n{df_train.columns.tolist()}")
print(f"\nClass distribution trong train:\n{df_train['Class'].value_counts().sort_index()}")
print(f"\nClass distribution trong validation:\n{df_val['Class'].value_counts().sort_index()}")

Khai báo dữ liệu
THÔNG TIN DỮ LIỆU
Train shape: (11516, 20)
Validation shape: (2880, 17)

Train columns:
['Id', 'Artist Name', 'Track Name', 'Popularity', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'time_signature', 'Class', 'key_sin', 'key_cos']

Class distribution trong train:
Class
0      400
1      878
2      814
3      258
4      248
5      926
6     1655
7      369
8     1186
9     1615
10    3167
Name: count, dtype: int64

Class distribution trong validation:
Class
0     100
1     220
2     204
3      64
4      62
5     231
6     414
7      92
8     297
9     404
10    792
Name: count, dtype: int64


In [3]:
# Cell 3: So sánh phân bố class
train_dist = df_train['Class'].value_counts(normalize=True).sort_index().rename('train_%')
val_dist = df_val['Class'].value_counts(normalize=True).sort_index().rename('val_%')
print(pd.concat([train_dist, val_dist], axis=1).round(4))

       train_%   val_%
Class                 
0       0.0347  0.0347
1       0.0762  0.0764
2       0.0707  0.0708
3       0.0224  0.0222
4       0.0215  0.0215
5       0.0804  0.0802
6       0.1437  0.1438
7       0.0320  0.0319
8       0.1030  0.1031
9       0.1402  0.1403
10      0.2750  0.2750


In [4]:
# Cell 4: So sánh mean của từng feature
drop_cols = ['Id', 'Artist Name', 'Track Name']
existing_drop_train = [col for col in drop_cols if col in df_train.columns]
existing_drop_val = [col for col in drop_cols if col in df_val.columns]

X_train = df_train.drop(columns=['Class'] + existing_drop_train)
y_train = df_train['Class']

X_val = df_val.drop(columns=['Class'] + existing_drop_val)
y_val = df_val['Class']

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")

# So sánh mean
compare = pd.DataFrame({
    'train_mean': X_train.mean(),
    'val_mean': X_val.mean(),
    'diff_pct': ((X_train.mean() - X_val.mean()) / X_train.mean() * 100).round(2)
})
print(compare.sort_values('diff_pct', key=abs, ascending=False))

X_train shape: (11516, 16)
X_val shape: (2880, 16)
                     train_mean       val_mean  diff_pct
key_sin                0.010156       0.025233   -148.45
instrumentalness       0.135972       0.132591      2.49
mode                   0.637635       0.650694     -2.05
valence                0.484816       0.492630     -1.61
key_cos               -0.186101      -0.188528     -1.30
Popularity            44.629298      44.048264      1.30
key                    6.083015       6.021528      1.01
duration_ms       234063.474870  232018.650311      0.87
danceability           0.543655       0.540908      0.51
liveness               0.195609       0.196473     -0.44
time_signature         3.927058       3.913542      0.34
loudness              -7.864334      -7.886753     -0.29
speechiness            0.080222       0.080017      0.26
tempo                122.739634     122.510843      0.19
energy                 0.662289       0.662952     -0.10
acousticness           0.246707      

In [5]:
# Cell 5: Train XGBoost model
print("\n" + "="*55)
print("TRAINING XGBOOST MODEL")
print("="*55)

# Tính sample weights để xử lý mất cân bằng class
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=12,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    min_child_weight=3,
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss',
    use_label_encoder=False,
    verbosity=0
)

xgb.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print(f"✅ Train xong!")
print(f"   n_estimators : {xgb.n_estimators}")
print(f"   max_depth    : {xgb.max_depth}")
print(f"   learning_rate: {xgb.learning_rate}")


TRAINING XGBOOST MODEL
✅ Train xong!
   n_estimators : 300
   max_depth    : 12
   learning_rate: 0.05


In [ ]:
# Cell 6: GridSearchCV để tìm hyperparameters tối ưu (TÙY CHỌN - có thể mất thời gian)
print("\n" + "="*55)
print("🔍 BẮT ĐẦU GRIDSEARCH CV CHO XGBOOST")
print("="*55)
print("⏱️ Quá trình này có thể mất vài phút...")

param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [8, 12, 16],
    'learning_rate': [0.03, 0.05, 0.07],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]
}

xgb_tuned = XGBClassifier(
    random_state=20,
    n_jobs=-1,
    eval_metric='mlogloss',
    use_label_encoder=False,
    verbosity=0
)

grid_search = GridSearchCV(
    xgb_tuned,
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train, sample_weight=sample_weights)

print("\n" + "="*55)
print("KẾT QUẢ GRIDSEARCH")
print("="*55)
print("Best params:", grid_search.best_params_)
print("Best CV F1 macro (5-fold):", grid_search.best_score_)

# Đánh giá với best model
best_xgb = grid_search.best_estimator_
y_pred_val_tuned = best_xgb.predict(X_val)

print("\n=== SAU KHI TUNE (dùng best model từ GridSearch) ===")
print(f"Accuracy  Validation : {accuracy_score(y_val, y_pred_val_tuned):.4f}")
print(f"Macro F1  Validation : {f1_score(y_val, y_pred_val_tuned, average='macro', zero_division=0):.4f}")
print("\nClassification Report trên Validation:")
print(classification_report(y_val, y_pred_val_tuned, zero_division=0))


🔍 BẮT ĐẦU GRIDSEARCH CV CHO XGBOOST
⏱️ Quá trình này có thể mất vài phút...
Fitting 5 folds for each of 162 candidates, totalling 810 fits


In [ ]:
# Cell 7: GridSearchCV để tìm hyperparameters tối ưu (tùy chọn, có thể chạy lâu)
print("\n" + "="*55)
print("🔍 BẮT ĐẦU GRIDSEARCH CV CHO LIGHTGBM")
print("="*55)
print("⏱️ Quá trình này có thể mất vài phút...")

param_grid = {
    'n_estimators': [200, 300, 500],
    'max_depth': [8, 12, 16],
    'learning_rate': [0.03, 0.05, 0.07],
    'num_leaves': [31, 63],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]
}

lgb_tuned = LGBMClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

grid_search = GridSearchCV(
    lgb_tuned,
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("\n" + "="*55)
print("KẾT QUẢ GRIDSEARCH")
print("="*55)
print("Best params:", grid_search.best_params_)
print("Best CV F1 macro (5-fold):", grid_search.best_score_)

# Đánh giá với best model
best_lgb = grid_search.best_estimator_
y_pred_val_tuned = best_lgb.predict(X_val)

print("\n=== SAU KHI TUNE (dùng best model từ GridSearch) ===")
print(f"Accuracy  Validation : {accuracy_score(y_val, y_pred_val_tuned):.4f}")
print(f"Macro F1  Validation : {f1_score(y_val, y_pred_val_tuned, average='macro', zero_division=0):.4f}")
print("\nClassification Report trên Validation:")
print(classification_report(y_val, y_pred_val_tuned, zero_division=0))

In [ ]:
# Cell 8: Dự đoán và đánh giá
print("\n" + "="*55)
print("LIGHTGBM - PREDICT & EVALUATE")
print("="*55)

y_pred_train = lgb.predict(X_train)
y_pred_val = lgb.predict(X_val)

acc_train = accuracy_score(y_train, y_pred_train)
acc_val = accuracy_score(y_val, y_pred_val)

print("ACCURACY")
print("-"*30)
print(f"  Train      : {acc_train:.4f} ({acc_train*100:.2f}%)")
print(f"  Validation : {acc_val:.4f} ({acc_val*100:.2f}%)")
print(f"  Gap        : {acc_train - acc_val:.4f}  ", end="")
if acc_train - acc_val > 0.1:
    print("⚠️ Có dấu hiệu Overfitting")
elif acc_val < 0.5:
    print("⚠️ Model yếu — Underfitting")
else:
    print("✅ Ổn định")

In [ ]:
# Cell 9: Classification Report chi tiết
print("\n" + "="*55)
print("CLASSIFICATION REPORT — TRAIN")
print("="*55)
print(classification_report(y_train, y_pred_train, zero_division=0))

print("="*55)
print("CLASSIFICATION REPORT — VALIDATION")
print("="*55)
print(classification_report(y_val, y_pred_val, zero_division=0))

In [ ]:
# Cell 10: Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, y_true, y_pred, title in zip(
    axes,
    [y_train, y_val],
    [y_pred_train, y_pred_val],
    ['Confusion Matrix — TRAIN (LightGBM)', 'Confusion Matrix — VALIDATION (LightGBM)']
):
    cm = confusion_matrix(y_true, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=sorted(y_train.unique()),
                yticklabels=sorted(y_train.unique()),
                linewidths=0.5, linecolor='gray')
    ax.set_title(title, fontweight='bold', fontsize=13)
    ax.set_xlabel('Predicted Class', fontsize=11)
    ax.set_ylabel('Actual Class', fontsize=11)

plt.suptitle('LightGBM — Confusion Matrix', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 11: Per-class F1 Score
classes = sorted(y_train.unique())
f1_train = f1_score(y_train, y_pred_train, average=None, labels=classes, zero_division=0)
f1_val = f1_score(y_val, y_pred_val, average=None, labels=classes, zero_division=0)

x = np.arange(len(classes))
width = 0.35

fig, ax = plt.subplots(figsize=(13, 5))
bars1 = ax.bar(x - width/2, f1_train, width, label='Train', color='#2ecc71', alpha=0.85)
bars2 = ax.bar(x + width/2, f1_val, width, label='Validation', color='#e67e22', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([f'Class {c}' for c in classes])
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1.1)
ax.set_title('F1 Score per Class — Train vs Validation (LightGBM)', fontweight='bold', fontsize=13)
ax.legend()
ax.axhline(0.7, color='gray', linestyle='--', linewidth=1, label='threshold 0.7')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

print("Class yếu nhất trên Validation (F1 < 0.7):")
for c, f1 in zip(classes, f1_val):
    if f1 < 0.7:
        print(f"  Class {c:>2d}: F1 = {f1:.4f}")

In [ ]:
# Cell 12: Feature Importance (LightGBM)
feat_imp = pd.Series(lgb.feature_importances_, index=X_train.columns)
feat_imp = feat_imp.sort_values(ascending=False)
feat_imp_norm = feat_imp / feat_imp.sum()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
feat_imp_norm.plot(kind='bar', ax=axes[0], color='#2ecc71', edgecolor='white', alpha=0.85)
axes[0].set_title('Feature Importance — LightGBM', fontweight='bold')
axes[0].set_ylabel('Normalized Importance Score')
axes[0].set_xticklabels(feat_imp_norm.index, rotation=45, ha='right')
for i, v in enumerate(feat_imp_norm.values):
    axes[0].text(i, v + 0.001, f'{v:.3f}', ha='center', fontsize=8)

# Cumulative importance
cumsum = feat_imp_norm.cumsum()
axes[1].plot(range(1, len(cumsum)+1), cumsum.values, marker='o', color='#e67e22')
axes[1].axhline(0.9, color='gray', linestyle='--', label='90% threshold')
axes[1].axhline(0.95, color='orange', linestyle='--', label='95% threshold')
axes[1].set_title('Cumulative Feature Importance (LightGBM)', fontweight='bold')
axes[1].set_xlabel('Number of Features')
axes[1].set_ylabel('Cumulative Importance')
axes[1].set_xticks(range(1, len(cumsum)+1))
axes[1].set_xticklabels(feat_imp_norm.index, rotation=45, ha='right')
axes[1].legend()

plt.suptitle('Feature Importance Analysis - LightGBM', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop 5 features quan trọng nhất (LightGBM):")
for feat, score in feat_imp_norm.head(5).items():
    print(f"  {feat:25s}: {score:.4f}")

n90 = (cumsum < 0.9).sum() + 1
print(f"\n→ Chỉ cần {n90} features để giải thích 90% importance")

In [ ]:
# Cell 13: Tổng kết đánh giá LightGBM
macro_f1_train = f1_score(y_train, y_pred_train, average='macro', zero_division=0)
macro_f1_val = f1_score(y_val, y_pred_val, average='macro', zero_division=0)
weighted_f1_val = f1_score(y_val, y_pred_val, average='weighted', zero_division=0)

print("╔══════════════════════════════════════════════╗")
print("║        TỔNG KẾT — LIGHTGBM MODEL            ║")
print("╠══════════════════════════════════════════════╣")
print(f"║  Accuracy   Train      : {acc_train:.4f}               ║")
print(f"║  Accuracy   Validation : {acc_val:.4f}               ║")
print(f"║  Macro F1   Train      : {macro_f1_train:.4f}               ║")
print(f"║  Macro F1   Validation : {macro_f1_val:.4f}               ║")
print(f"║  Weighted F1 Validation: {weighted_f1_val:.4f}               ║")
print(f"║  Overfitting gap       : {acc_train - acc_val:.4f}               ║")
print("╠══════════════════════════════════════════════╣")

gap = acc_train - acc_val
if gap > 0.1:
    print("║  ⚠️  OVERFITTING — cần giảm độ phức tạp     ║")
    print("║     → Thử: max_depth=8~12, reg_alpha/lambda  ║")
elif acc_val < 0.5:
    print("║  ⚠️  UNDERFITTING — model chưa học được      ║")
    print("║     → Thử: tăng n_estimators, giảm learning_rate ║")
else:
    print("║  ✅  Model ổn định                           ║")

print("╚══════════════════════════════════════════════╝")

In [ ]:
# Cell 14: So sánh LightGBM vs Random Forest
print("\n" + "="*55)
print("📊 SO SÁNH LIGHTGBM VS RANDOM FOREST")
print("="*55)

print("\n{:<20} {:>15} {:>15} {:>15}".format("Metric", "Random Forest", "LightGBM", "Chênh lệch"))
print("-"*65)

# Random Forest metrics từ file gốc
rf_acc_train = 0.9168
rf_acc_val = 0.4524
rf_macro_f1 = 0.4459

metrics = [
    ("Accuracy Train", rf_acc_train, acc_train),
    ("Accuracy Val", rf_acc_val, acc_val),
    ("Macro F1 Val", rf_macro_f1, macro_f1_val),
]

for name, rf_val, lgb_val in metrics:
    diff = lgb_val - rf_val
    diff_str = f"+{diff:.4f}" if diff > 0 else f"{diff:.4f}"
    arrow = "⬆️" if diff > 0 else "⬇️" if diff < 0 else "➡️"
    print("{:<20} {:>15.4f} {:>15.4f} {:>15} {}".format(name, rf_val, lgb_val, diff_str, arrow))

print("\n" + "="*55)
if acc_val > rf_acc_val:
    print("✅ LightGBM outperforms Random Forest on Validation!")
elif acc_val < rf_acc_val:
    print("⚠️ Random Forest still performs better on Validation")
else:
    print("➡️ Both models perform similarly")